# FinSight · P2 Analysis/NLP baseline
**Owner: Nathan · Sprint 1 · A7, A2 preparation, W1 handoff**

Run from top to bottom. This notebook imports reusable backend code and needs no SEC or LLM key. It demonstrates source-preserving sentence segmentation, spaCy NER, company normalization, cited evidence notes, and a human annotation queue.

**Implemented:** NER and evidence contracts. **Still to do:** human sentiment annotations, sentiment inference/fine-tuning, financial metrics, and P4/P3 integration. Evidence notes below are original excerpts, not generated financial analysis.

The three real SEC excerpts are a small starter queue, not a representative sentiment test set. Synthetic NER examples are separate and never exported into that queue.

## 1. Environment
From the repository root, with Python 3.11 or 3.12:
```bash
python3.12 -m venv .venv-nlp
source .venv-nlp/bin/activate
pip install -r ACSESS/finsight/requirements-nlp.txt
python -m spacy download en_core_web_sm
```
Select `.venv-nlp/bin/python` as this notebook's kernel. In Colab, first clone/upload the repository, install the same requirements, and set `REPO_ROOT_OVERRIDE` below. Model downloads are setup steps; the analysis itself runs offline.

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT_OVERRIDE = None  # e.g. Path('/content/AI-Stock-Research-and-Earnings-Assistant')
if REPO_ROOT_OVERRIDE is not None:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve()
else:
    REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / 'ACSESS/finsight/backend/app/nlp').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from the cloned repo, or set REPO_ROOT_OVERRIDE.')
APP_ROOT = REPO_ROOT / 'ACSESS/finsight'
sys.path.insert(0, str(APP_ROOT))

import pandas as pd
from backend.app.agents.analysis import analyze
from backend.app.nlp.annotations import annotation_summary, export_annotations, save_result
from backend.app.nlp.entities import build_pipeline
from backend.app.nlp.evaluation import evaluate_entities
from backend.app.nlp.schemas import Passage
print(f'Repository: {REPO_ROOT.name}')

Repository: AI-Stock-Research-and-Earnings-Assistant


## 2. Stored passages with citations
Replace `PASSAGES_PATH` with a JSON list of your team's stored excerpts using the same schema. Keep source URL, accession, form, reporting period, and section. Do not assign HTML page numbers: offsets below refer only to the supplied clean passage text.

Sources: NVIDIA [FY2025 10-K](https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm), [FY2026 10-K](https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm), and [July 2025 10-Q](https://www.sec.gov/Archives/edgar/data/1045810/000104581025000209/nvda-20250727.htm). Nonbreaking spaces in the samples are normalized to spaces; text is otherwise excerpted verbatim.

In [2]:
PASSAGES_PATH = APP_ROOT / 'backend/tests/fixtures/nlp/passages.json'
passages = [Passage.model_validate(row) for row in json.loads(PASSAGES_PATH.read_text())]
display(pd.DataFrame([p.model_dump(mode='json') for p in passages]))

,ticker,source_type,source_url,accession,form,period_end,section,text
0,NVDA,filing,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-25-000023,10-K,2025-01-26,Item 7 — Overview,Revenue growth in fiscal year 2025 was driven ...
1,NVDA,filing,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-26-000021,10-K,2026-01-25,"Item 7 — Recent Developments, Future Objective...",We generated approximately $60 million in H20 ...
2,NVDA,filing,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-25-000209,10-Q,2025-07-27,"Part I, Item 2 — Macroeconomic Factors",Our product and solution pricing generally doe...


## 3. Sentence extraction and NER
Default: spaCy `en_core_web_sm` plus explicit company alias/ticker rules. Set `MODEL_NAME = None` to inspect the rules-only baseline. Missing models raise an error instead of silently changing the experiment.

Generic ORG predictions may include suppliers or competitors. `ticker` identifies the passage's subject; `canonical_ticker` identifies an individual recognized company. They need not match. NER spans have no invented confidence scores.

In [3]:
MODEL_NAME = 'en_core_web_sm'
nlp = build_pipeline(MODEL_NAME)
result = analyze(passages, nlp)
print(result.pipeline)
statement_rows = []
entity_rows = []
for statement in result.statements:
    statement_rows.append({'statement_id': statement.statement_id, 'ticker': statement.ticker,
                           'text': statement.text, 'entities': len(statement.entities)})
    for entity in statement.entities:
        entity_rows.append({'statement_id': statement.statement_id, **entity.model_dump()})
display(pd.DataFrame(statement_rows))
display(pd.DataFrame(entity_rows))

spacy:3.8.16:core_web_sm:3.8.0


,statement_id,ticker,text,entities
0,S33c9b66c2c6b984f,NVDA,Revenue growth in fiscal year 2025 was driven ...,2
1,S811766621a9d26e0,NVDA,We generated approximately $60 million in H20 ...,2
2,Sbfa6cc229387179c,NVDA,Our product and solution pricing generally doe...,0


,statement_id,text,label,start,end,canonical_ticker
0,S33c9b66c2c6b984f,fiscal year 2025,DATE,18,34,None
1,S33c9b66c2c6b984f,AI,ORG,124,126,None
2,S811766621a9d26e0,approximately $60 million,MONEY,13,38,None
3,S811766621a9d26e0,H20,GPE,42,45,None


## 4. Evidence-note contract for Report and shared memory
Each `N…` note cites one `S…` statement. Resolve that statement's `passage_id` to retrieve filing metadata and the original excerpt. IDs are stable across reruns; a changed passage receives a new ID.

These are cited evidence excerpts. Semantic claim verification and research report generation remain future work.

In [4]:
statements_by_id = {s.statement_id: s for s in result.statements}
note_rows = []
for note in result.notes:
    statement = statements_by_id[note.source_ids[0]]
    source = result.sources[statement.passage_id]
    note_rows.append({'note_id': note.note_id, 'source_id': statement.statement_id,
                      'text': note.text, 'section': source.section,
                      'accession': source.accession, 'source_url': str(source.source_url)})
display(pd.DataFrame(note_rows))
RESULT_PATH = REPO_ROOT / 'data/nlp/analysis.json'
save_result(result, RESULT_PATH)
print('Saved:', RESULT_PATH.relative_to(REPO_ROOT))

,note_id,source_id,text,section,accession,source_url
0,N71dc604de2112911,S33c9b66c2c6b984f,Revenue growth in fiscal year 2025 was driven ...,Item 7 — Overview,0001045810-25-000023,https://www.sec.gov/Archives/edgar/data/104581...
1,Nbef528b7ae743d40,S811766621a9d26e0,We generated approximately $60 million in H20 ...,"Item 7 — Recent Developments, Future Objective...",0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...
2,N9ad9499a90b07136,Sbfa6cc229387179c,Our product and solution pricing generally doe...,"Part I, Item 2 — Macroeconomic Factors",0001045810-25-000209,https://www.sec.gov/Archives/edgar/data/104581...


Saved: data/nlp/analysis.json


## 5. Human sentiment annotation queue
Read each original source and annotate sentiment **toward the target company**, rather than copying a model's answer.

- `gold_sentiment`: positive, neutral, negative, uncertain, or not_applicable.
- `gold_reasons`: semicolon-separated category IDs from the annotation guide.
- `annotator`: your name; `review_status`: unlabeled, labeled, or reviewed.
- `annotation_notes`: context, rationale, or disagreements.

Export merges by stable IDs, keeps older rows, and preserves existing human fields. No gold sentiment labels are generated automatically. Keep future train/dev/test partitions separate by filing and check duplicate text across partitions. See `docs/nlp_annotation_guide.md`.

In [5]:
ANNOTATIONS_PATH = REPO_ROOT / 'datasets/nlp/annotation_queue.csv'
export_annotations(result, ANNOTATIONS_PATH)
display(pd.read_csv(ANNOTATIONS_PATH, keep_default_na=False))
print(annotation_summary(ANNOTATIONS_PATH))

,statement_id,ticker,text,source_url,accession,form,period_end,section,passage_id,start,end,gold_sentiment,gold_reasons,annotator,review_status,annotation_notes
0,S33c9b66c2c6b984f,NVDA,Revenue growth in fiscal year 2025 was driven ...,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-25-000023,10-K,2025-01-26,Item 7 — Overview,Pf62d510a7280a4ae,0,137,,,,unlabeled,
1,S811766621a9d26e0,NVDA,We generated approximately $60 million in H20 ...,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-26-000021,10-K,2026-01-25,"Item 7 — Recent Developments, Future Objective...",P986eef2b96763681,0,75,,,,unlabeled,
2,Sbfa6cc229387179c,NVDA,Our product and solution pricing generally doe...,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-25-000209,10-Q,2025-07-27,"Part I, Item 2 — Macroeconomic Factors",P5f4e5f035749e5b9,0,99,,,,unlabeled,


{'total': 3, 'labeled': 0, 'reviewed': 0, 'usable_reviewed': 0}


## 6. NER evaluation on synthetic smoke examples
The expected entity spans are manually authored for three synthetic sentences. This catches pipeline errors and exposes the rules-only baseline's limited coverage. It is **not** a claim of accuracy on real filings. Add representative human-annotated filing passages before reporting project NER F1.

In [6]:
examples = json.loads((APP_ROOT / 'backend/tests/fixtures/nlp/ner_gold.json').read_text())
eval_passages = []
gold = {}
for example in examples:
    passage = Passage(ticker='NVDA', source_type='synthetic', section='Synthetic NER smoke example',
                      text=example['text'])
    eval_passages.append(passage)
    spans = []
    for entity in example['entities']:
        start = passage.text.index(entity['text'])
        spans.append({'start': start, 'end': start + len(entity['text']), 'label': entity['label']})
    gold[passage.passage_id] = spans
scores = []
for label, pipeline in [('rules_only', build_pipeline(None)), ('selected_pipeline', nlp)]:
    evaluated = analyze(eval_passages, pipeline)
    score = evaluate_entities(evaluated, gold)
    scores.append({'pipeline': label, **{k: v for k, v in score.items() if k not in ('missed', 'extra')}})
    print(label, 'missed:', score['missed'], 'extra:', score['extra'])
display(pd.DataFrame(scores))

rules_only missed: [('P69ff1b3633f0e009', 0, 13, 'ORG')] extra: []
selected_pipeline missed: [('P69ff1b3633f0e009', 0, 13, 'ORG')] extra: []


,pipeline,precision,recall,f1,matched,predicted,gold
0,rules_only,1.0,0.875,0.933333,7,7,8
1,selected_pipeline,1.0,0.875,0.933333,7,7,8


## 7. Sprint checklist
- [x] Runnable offline NER baseline with source-linked output.
- [x] P2-to-P3/P4 evidence-note contract and regression tests.
- [x] Human annotation queue and guidelines; labels remain blank.
- [ ] Expand to ~200 real statements across the agreed companies and sources.
- [ ] Nathan labels statements; a teammate reviews a subset.
- [ ] Annotate real filing entity spans and measure NER F1.
- [ ] Add FinBERT inference and then compare/fine-tune on a separate evaluation set.
- [ ] Add XBRL metrics and shared-memory integration.

This notebook proves the starter pipeline runs. Human annotations, real-world accuracy, and the complete Analysis agent remain to be completed.